1) Import libraries

In [1]:
import pandas as pd, pyarrow.parquet as pq, pyarrow.csv as csv
import re, os, io, duckdb, requests


2) Set Script Parameters

In [2]:
# Pick the years to include in the script
Years = range(2023,2026,1)

# Enter the parameters used in loading and storing files from the predownloaded process
predownloaded_directory = "monthly_datasets\\"
predownloaded_prefix = "hourly_transportation_"
predownloaded_suffix = ".csv" 
predownload_export = "export\\predownload\\"

# Enter the parameters used in loading and storing filed from the web-retireval process
query_list_file = "New Microsoft Excel Worksheet.xlsx"
query_export = "export\\query\\"

# Enter the column names you wish to keep in your export, and which columns to aggregate accross
kept_columns = "transition_date, transition_hour, number_of_passage, number_of_passenger, product_kind, transaction_type_desc, town, line_name, station_poi_desc_cd"
spread_columns = "number_of_passage, number_of_passenger"

# Enter the parameters used in naming and storing the output files
export_prefix = "hourly_transportation_"
export_suffix = ".parquet"
export_format = "PARQUET"

In [3]:
### DEPRACATED: DUCKDB METHOD IS UP TO 466x FASTER

# A function to remove extra columns and aggregate accross numeric columns which are spread out vertically
# ex:   "Vehicle" "Passangers"      "Vehicle" "Passangers"
#       Car         1           ->   Car        4
#       Car         3

# def clean_dataset(dataset:pd.DataFrame):
#     # Fetch a list of columns to keep
#     grouping_cols = [col_name.strip(", ") for col_name in kept_columns.split()]
#     # Fetch a list of columns to aggregate accross by sum
#     agging_dict = {col_name:sum for col_name in spread_columns.split(", ")}
#     # Ensure the grouping columns dont include the aggregation colums
#     [grouping_cols.remove(col) for col in agging_dict.keys()]

    
#     # Group by identity columns and compress the spread out information
#     df_combined = dataset.groupby(grouping_cols, as_index=False, dropna= False).agg(agging_dict)
    
#     return df_combined

# file = pd.read_csv("monthly_datasets\\hourly_transportation_202410.csv")

# cleaning_test = clean_dataset(file)

In [14]:
# A function to remove extra columns and aggregate accross numeric columns which are spread out vertically
# ex:   "Vehicle" "Passangers"      "Vehicle" "Passangers"
#       Car         1           ->   Car        4
#       Car         3

def clean_dataset(dataset:duckdb.DuckDBPyRelation):
    # Fetch a list of columns to keep
    grouping_cols = [col_name.strip(", ") for col_name in kept_columns.split()]

    # Fetch a list of columns to aggregate accross by sum
    agging_dict = {col_name.strip(", "):"sum" for col_name in spread_columns.split()}
    
    # Ensure the grouping columns dont include the aggregation colums
    [grouping_cols.remove(col) for col in agging_dict.keys()]

    # Group by identity columns and compress the spread out information
    # df_combined = dataset.aggregate(
    # aggr_expr = f"{', '.join([f'{agg_fun}({col})' for col, agg_fun in agging_dict.items()])}",
    # group_expr= f"{', '.join(grouping_cols)}")

    # ALternate method of aggregation using an SQL query
    df_combined = duckdb.sql(f"""
        SELECT {", ".join(grouping_cols+[f"sum(TRY({col}::INTEGER)) as {col}" for col in spread_columns.split(", ")])}
        FROM dataset
        GROUP BY {", ".join(grouping_cols)}""")
    return df_combined


# test prompt
# file = duckdb.read_csv("monthly_datasets\\hourly_transportation_202410.csv")
# cleaning_test = clean_dataset(file)
# cleaning_test

3) Gather filtered lists of files

In [5]:
# Fetch the spreadsheet with resource info in it
query_sheet = pd.read_excel(query_list_file)
# Filter resource sheet by selected years
filtered_query_sheet = query_sheet[query_sheet.Year.isin(Years)]

# Fetch the list of files that were manually saved as {predownloaded_suffix} for the selected years
predownloaded_file_directory_list = []
for year in Years:
    filename_pattern = f'{predownloaded_directory}{predownloaded_prefix}*{year}*{predownloaded_suffix}' 
    predownloaded_file_directory_list.extend([r[0] for r in duckdb.sql(f"SELECT * FROM glob('{filename_pattern}')").fetchall()])


4) Query the files and store them in compressed format

In [15]:
# Function to fetch CSV files, using a download link, process it, and export it
def get_from_CSV(file_address):

    ### Ensure given address can only be a string
    if type(file_address)!= str:
        print("file_address must be a string")
        raise TypeError
    
    ### extract file name and data
    try:
        print(file_address,"... ",sep="", end=" ")

        # load the csv data into a duckdb relation table
        file_db = duckdb.read_csv(file_address, filename=False)

        # extract file name from the address
        file_name = file_address.split("\\")[-1].split("/")[-1]  #alternate incase filename=True duckdb.sql("SELECT filename from file_db LIMIT 1").fetchall()[0][0]

    # terminate processing if file not found
    except IOError as error:
        print("Failed! File not found?")
        print(IOError)
        return False
    
    # terminate processing if unexpected error occurs
    except Exception as Error:
        print("Failed!")
        print("An unexpected error has occured in reading the file: ", Error)
        return False
    

    ### Positive response initiates export name generation
    # incase only YYYYMM is used in export naming
    # try:           
    #     # extract year and month from file name
    #     match = re.search(r"(\d{4})(\d{2})",file_name)
    #     Year = match.group(1)
    #     Month_number = match.group(2)
        
    #     # generate naming id using year and month
    #     export_id = f'{Year}{Month_number}'

    # except:
    #     print("\"YYYYMM\" id format was not found in: ", file_name,end="   ")
    #     match = re.findall(f"{predownloaded_prefix}(*){predownloaded_suffix}", file_name)
    #     export_id = match[0] if match != [] else ""
    
    # get a name id mimic for the export file 
    match = re.findall(f"{predownloaded_prefix}(.*){predownloaded_suffix}", file_name)
    export_id = match[0] if match != [] else ""
    # generate the export address 
    export_address = f"{predownload_export}{export_prefix}{export_id}{export_suffix}"

    ### clean the duckdb table
    try:
        print("cleaning...  ",end="")
        cleaned_db = clean_dataset(file_db)
    except Exception as Error:
        print("Failed!")
        print("An error has occured in cleaning the table: ", Error)
        return False

    ### export the cleaned table
    try:
        # attempt export using duckdb
        print("exporting via duckdb...  ",end="")
        duckdb.sql(f"""COPY 
            (SELECT {kept_columns} from cleaned_db) 
            TO "{export_address}"
            (FORMAT {export_format})""")
        
    except Exception as Error:
        print("Failed!")
        print("An error has occured in exporting using duckdb: ", Error)

        # remove partially saved/corrupted files
        if os.path.exists(export_address):
            os.remove(export_address)
            print("Cleaned up partial/corrupted file.")
        try:
            print("exporting via pyarrows...  ",end="")

            # 1. Fetch the query as an Arrow RecordBatchReader (streams data, doesn't load it all)
            reader = duckdb.sql(f"SELECT {kept_columns} FROM cleaned_db").to_arrow_reader()

            # 2. Stream it straight into a Parquet or CSV file
            if export_format.lower() in ['parquet', 'pq']:
                pq.write_table(reader.read_all(), f"{predownload_export}{export_prefix}{export_id}(test){export_suffix}")
            elif export_format.lower() in ['csv', 'txt']:
                csv.write_csv(reader.read_all(), f"{predownload_export}{export_prefix}{export_id}(test){export_suffix}")
            else:
                raise ValueError(f"Unsupported file format: {export_format}")
            
        except Exception as Error:
            print("Failed!")
            print("An error has occured in exporting using pyarrows: ", Error)

            # remove partially saved/corrupted files
            if os.path.exists(export_address):
                os.remove(export_address)
                print("Cleaned up partial/corrupted file.")
                return False
    # exit on completion
    print("Done!")
    return True

# import pyarrow.parquet as pq
# print(get_from_CSV("monthly_datasets\\hourly_transportation_202403.csv"))
gean = [get_from_CSV(file_address) for file_address in predownloaded_file_directory_list[-5:]]
print(gean, sep="\n")

monthly_datasets\hourly_transportation_202409.csv...  cleaning...  exporting via duckdb...  Done!
monthly_datasets\hourly_transportation_202410.csv...  cleaning...  exporting via duckdb...  Done!
monthly_datasets\hourly_transportation_202411.csv...  cleaning...  exporting via duckdb...  Done!
monthly_datasets\hourly_transportation_202412.csv...  cleaning...  exporting via duckdb...  Done!
monthly_datasets\hourly_transportation_202512.csv...  cleaning...  exporting via duckdb...  Done!
[True, True, True, True, True]


In [ ]:
# Function to fetch CSV files, using a download link, process it, and export it
def get_by_CSV(query):
    # query the source for the CSV File
    response = requests.get(query.CSV_URL)

    # If response is negative, print error message
    if response.status_code != 200:
        print(f'{query.Month} {query.Year} returned a code of {response.status_code}')
        return False
    
    # Positive response initiates file processing
    else:
        # generate naming id
        export_id = f'{query.Year}{query.Month_number:02d}'

        # load the csv data into a duckdb relation table and clean it
        pseudofile = clean_dataset(duckdb.read_csv(io.StringIO(response.text)))

        # export the cleaned table
        duckdb.sql(f"""COPY 
           (SELECT {kept_columns} from pseudofile) 
            TO "{query_export}{export_prefix}{export_id}{export_suffix}"
           (FORMAT {export_format})""")
        return True

gean = filtered_query_sheet.tail(3).apply(get_by_CSV, axis=1)

In [7]:
predownloaded_file_directory_list[-5:]

['monthly_datasets\\hourly_transportation_202409.csv',
 'monthly_datasets\\hourly_transportation_202410.csv',
 'monthly_datasets\\hourly_transportation_202411.csv',
 'monthly_datasets\\hourly_transportation_202412.csv',
 'monthly_datasets\\hourly_transportation_202512.csv']